# Lab Test Trends — Silver to Gold

Builds `adb_caresynce.gold.lab_test_trends` — one row per test type per calendar month with abnormal/critical rates and month-over-month trend.

**Source tables:** `silver.lab_results`, `silver.calendar`  
**Load strategy:** Full rebuild (CREATE OR REPLACE TABLE)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresynce.gold

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresynce.gold.lab_test_trends AS
WITH base AS (
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                                 AS year,
        MONTH(lr.test_date)                                                AS month,
        COUNT(*)                                                           AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)         AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)         AS critical_test_count
    FROM adb_caresynce.silver.lab_results lr
    GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
)
SELECT
    b.test_name,
    b.year,
    b.month,
    c.month_name,
    c.quarter_name,
    b.total_tests,
    b.abnormal_test_count,
    b.critical_test_count,
    CAST(ROUND(b.abnormal_test_count * 100.0 / b.total_tests, 2) AS DECIMAL(5,2))   AS abnormal_rate_pct,
    CAST(ROUND(b.critical_test_count * 100.0 / b.total_tests, 2) AS DECIMAL(5,2))   AS critical_rate_pct,
    CAST(
        LAG(ROUND(b.abnormal_test_count * 100.0 / b.total_tests, 2))
            OVER (PARTITION BY b.test_name ORDER BY b.year, b.month)
        AS DECIMAL(5,2)
    )                                                                                AS prev_month_abnormal_rate_pct,
    CAST(
        ROUND(b.abnormal_test_count * 100.0 / b.total_tests, 2)
        - LAG(ROUND(b.abnormal_test_count * 100.0 / b.total_tests, 2))
              OVER (PARTITION BY b.test_name ORDER BY b.year, b.month)
        AS DECIMAL(6,2)
    )                                                                                AS abnormal_rate_change_pct
FROM base b
LEFT JOIN adb_caresynce.silver.calendar c
    ON c.date_key = CAST(DATE_FORMAT(MAKE_DATE(b.year, b.month, 1), 'yyyyMMdd') AS INT)

In [0]:
%sql
SELECT * FROM adb_caresynce.gold.lab_test_trends LIMIT 20

In [0]:
updated_df = spark.table("adb_caresynce.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))